In [1]:
import pandas as pd

# Load one year to check the structure
df_test = pd.read_csv("../data/raw/BACI_HS17_V202601/BACI_HS17_Y2024_V202601.csv")
print(df_test.shape)
print(df_test.dtypes)
df_test.head()

(11177570, 6)
t      int64
i      int64
j      int64
k      int64
v    float64
q    float64
dtype: object


,t,i,j,k,v,q
0,2024,4,24,80810,0.176,0.100
1,2024,4,24,330499,2.295,0.230
2,2024,4,24,732510,5.617,0.038
3,2024,4,24,848330,2.420,0.004
4,2024,4,24,853610,0.605,0.011


In [2]:
# Filter only gold (HS code starts with 7108)
gold = df_test[df_test["k"].astype(str).str.startswith("7108")]
print(gold.shape)
gold.head()

(6423, 6)


,t,i,j,k,v,q
829,2024,4,124,71080,0.005,0.001
4091,2024,4,528,71080,1.052,0.430
7666,2024,4,860,710812,8535.959,NaN
9427,2024,8,100,71080,0.189,0.583
15081,2024,8,380,71080,115.236,61.780


In [3]:
import glob

# Load all years and filter gold
files = glob.glob("../data/raw/BACI_HS17_V202601/BACI_HS17_Y*.csv")

dfs = []
for f in files:
    df = pd.read_csv(f)
    gold = df[df["k"].astype(str).str.startswith("7108")]
    dfs.append(gold)

gold_all = pd.concat(dfs, ignore_index=True)
print(gold_all.shape)
print(gold_all["t"].value_counts().sort_index())

(50331, 6)
t
2017    5420
2018    5930
2019    6307
2020    6235
2021    6551
2022    6778
2023    6687
2024    6423
Name: count, dtype: int64


In [4]:
# Load country codes
countries = pd.read_csv("../data/raw/BACI_HS17_V202601/country_codes_V202601.csv")
print(countries.head())

   country_code    country_name country_iso2 country_iso3
0             4     Afghanistan           AF          AFG
1             8         Albania           AL          ALB
2            12         Algeria           DZ          DZA
3            16  American Samoa           AS          ASM
4            20         Andorra           AD          AND


In [5]:
# Add country names
gold_all = gold_all.merge(countries[["country_code", "country_name"]], left_on="i", right_on="country_code")
gold_all = gold_all.rename(columns={"country_name": "exporter"}).drop("country_code", axis=1)

gold_all = gold_all.merge(countries[["country_code", "country_name"]], left_on="j", right_on="country_code")
gold_all = gold_all.rename(columns={"country_name": "importer"}).drop("country_code", axis=1)

print(gold_all.shape)
gold_all.head()

(50331, 8)


,t,i,j,k,v,q,exporter,importer
0,2017,4,586,71080,1.278,1.534,Afghanistan,Pakistan
1,2017,8,70,71080,1.588,3.062,Albania,Bosnia Herzegovina
2,2017,8,100,71080,26.621,15.120,Albania,Bulgaria
3,2017,8,380,71080,19.410,27.750,Albania,Italy
4,2017,8,380,710812,212.456,0.006,Albania,Italy


In [6]:
# Check Switzerland
swiss = gold_all[(gold_all["exporter"] == "Switzerland") | (gold_all["importer"] == "Switzerland")]
print(swiss.shape)
print(swiss["t"].value_counts().sort_index())

(2821, 8)
t
2017    353
2018    335
2019    359
2020    350
2021    356
2022    374
2023    342
2024    352
Name: count, dtype: int64


In [7]:
# Save filtered gold data
gold_all.to_csv("../data/processed/baci_gold_2017_2024.csv", index=False)
swiss.to_csv("../data/processed/baci_switzerland_gold.csv", index=False)
print("saved!")

saved!


In [8]:
# Load old dataset
df_old = pd.read_csv("../data/processed/precious_metals_trade_inflation.csv")
print(df_old.columns.tolist())
print(df_old.head(2))

['country', 'year', 'commodity', 'metal_group', 'flow', 'trade_value_usd', 'weight_kg', 'category', 'Core CPI', 'Energy CPI', 'Food CPI', 'Headline CPI']
   country  year                                          commodity  \
0  Albania  2013  Precious metal ores and concentrates except si...   
1  Albania  2012  Precious metal ores and concentrates except si...   

  metal_group    flow  trade_value_usd  weight_kg              category  \
0      Silver  Export              401    29000.0  26_ores_slag_and_ash   
1      Silver  Import              668     3890.0  26_ores_slag_and_ash   

   Core CPI  Energy CPI  Food CPI  Headline CPI  
0  0.189061        0.24  4.240740      1.934749  
1  1.712044        0.90  2.340568      2.027813  


In [10]:
# Reshape BACI to match old dataset structure
baci_reshaped = gold_all.rename(columns={
    "t": "year",
    "j": "country_code",
    "v": "trade_value_usd",
    "q": "weight_kg",
    "importer": "country"
})

baci_reshaped["flow"] = "Import"
baci_reshaped["metal_group"] = "Gold"

# Keep only relevant columns
baci_reshaped = baci_reshaped[["country", "year", "metal_group", "flow", "trade_value_usd", "weight_kg"]]

print(baci_reshaped.shape)
baci_reshaped.head()

(50331, 6)


,country,year,metal_group,flow,trade_value_usd,weight_kg
0,Pakistan,2017,Gold,Import,1.278,1.534
1,Bosnia Herzegovina,2017,Gold,Import,1.588,3.062
2,Bulgaria,2017,Gold,Import,26.621,15.120
3,Italy,2017,Gold,Import,19.410,27.750
4,Italy,2017,Gold,Import,212.456,0.006


In [11]:
# Fix trade value (BACI is in thousand USD)
baci_reshaped["trade_value_usd"] = (baci_reshaped["trade_value_usd"] * 1000).astype(int)

print(baci_reshaped["trade_value_usd"].describe())

count    5.033100e+04
mean     7.416802e+07
std      6.466972e+08
min      1.000000e+00
25%      4.278500e+03
50%      6.030500e+04
75%      1.351160e+06
max      3.084176e+10
Name: trade_value_usd, dtype: float64


In [13]:
import os
print(os.listdir("../data/raw/"))

['BACI_HS17_V202601', 'Inflation-data.xlsx']


In [15]:
inflation = pd.read_excel("../data/raw/Inflation-data.xlsx")
print(inflation.shape)
print(inflation.columns.tolist()[:10])
inflation.head(2)

(25, 3)
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2']


,Unnamed: 0,Unnamed: 1,Unnamed: 2
0,NaN,NaN,NaN
1,NaN,NaN,NaN


In [16]:
inflation = pd.read_excel("../data/raw/Inflation-data.xlsx", sheet_name=None)
print(inflation.keys())

dict_keys(['Intro', 'top', 'hcpi_m', 'hcpi_q', 'hcpi_a', 'ecpi_m', 'ecpi_q', 'ecpi_a', 'fcpi_m', 'fcpi_q', 'fcpi_a', 'ccpi_m', 'ccpi_q', 'ccpi_a', 'ppi_m', 'ppi_q', 'ppi_a', 'def_q', 'def_a', 'Aggregate'])


In [17]:
inflation = pd.read_excel("../data/raw/Inflation-data.xlsx", sheet_name="hcpi_a", header=1)
print(inflation.shape)
print(inflation.columns.tolist()[:10])
inflation.head(2)

(204, 61)
['ABW', 314, 'Aruba', 'Inflation', 'Headline Consumer Price Inflation', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']


,ABW,314,Aruba,Inflation,Headline Consumer Price Inflation,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,-0.89,-0.474,3.58,4.257,1.222,0.744,5.52,3.363,1.712,Annual average inflation
0,AFG,512.0,Afghanistan,Inflation,Headline Consumer Price Inflation,25.51,25.51,-12.52,-10.68,10.23,...,4.380000,4.976,0.63,2.302,5.443,5.062000,10.600000,-7.714,-6.601186,Annual average inflation
1,AGO,614.0,Angola,Inflation,Headline Consumer Price Inflation,7.97,5.78,15.80,15.67,27.42,...,32.377731,29.844,19.63,17.079,21.024,23.846111,23.826819,13.639,28.240495,Annual average inflation


In [19]:
inflation_raw = pd.read_excel("../data/raw/Inflation-data.xlsx", sheet_name="hcpi_a", header=0)
print(inflation_raw.iloc[:3, :8])

  Country Code  IMF Country Code      Country Indicator Type  \
0          ABW             314.0        Aruba      Inflation   
1          AFG             512.0  Afghanistan      Inflation   
2          AGO             614.0       Angola      Inflation   

                         Series Name   1970   1971   1972  
0  Headline Consumer Price Inflation    NaN    NaN    NaN  
1  Headline Consumer Price Inflation  25.51  25.51 -12.52  
2  Headline Consumer Price Inflation   7.97   5.78  15.80  


In [20]:
# Reshape from wide to long
inflation_long = pd.melt(
    inflation_raw,
    id_vars=["Country Code", "Country"],
    value_vars=[col for col in inflation_raw.columns if str(col).isdigit()],
    var_name="year",
    value_name="Headline CPI"
)

inflation_long["year"] = inflation_long["year"].astype(int)
inflation_long = inflation_long.rename(columns={"Country Code": "country_iso3", "Country": "country"})
inflation_long = inflation_long.dropna(subset=["Headline CPI"])

print(inflation_long.shape)
inflation_long.head()

(9912, 4)


,country_iso3,country,year,Headline CPI
1,AFG,Afghanistan,1970,25.510000
2,AGO,Angola,1970,7.970000
4,ARE,United Arab Emirates,1970,21.984699
5,ARG,Argentina,1970,13.586900
7,ATG,Antigua and Barbuda,1970,8.870000


In [21]:
# Filter 2017-2024
inflation_recent = inflation_long[inflation_long["year"].between(2017, 2024)]
print(inflation_recent.shape)

# Join with BACI data using country name + year
baci_with_inflation = baci_reshaped.merge(
    inflation_recent[["country", "year", "Headline CPI"]],
    on=["country", "year"],
    how="left"
)

print(baci_with_inflation.shape)
baci_with_inflation.head()

(1591, 4)
(50331, 7)


,country,year,metal_group,flow,trade_value_usd,weight_kg,Headline CPI
0,Pakistan,2017,Gold,Import,1278,1.534,4.148
1,Bosnia Herzegovina,2017,Gold,Import,1588,3.062,NaN
2,Bulgaria,2017,Gold,Import,26621,15.120,1.188
3,Italy,2017,Gold,Import,19410,27.750,1.326
4,Italy,2017,Gold,Import,212456,0.006,1.326


In [22]:
baci_with_inflation.to_csv("../data/processed/baci_gold_with_inflation.csv", index=False)
print("saved!")
print(baci_with_inflation["Headline CPI"].isna().sum(), "missing CPI values")

saved!
8921 missing CPI values


In [23]:
# Check which countries don't match
missing = baci_with_inflation[baci_with_inflation["Headline CPI"].isna()]["country"].unique()
print(len(missing), "countries without CPI match")
print(sorted(missing)[:20])

69 countries without CPI match
['American Samoa', 'Andorra', 'Anguilla', 'Bahamas', 'Bermuda', 'Bolivia (Plurinational State of)', 'Bonaire', 'Bosnia Herzegovina', 'Br. Virgin Isds', 'Cayman Isds', 'Central African Rep.', 'China, Hong Kong SAR', 'China, Macao SAR', 'Christmas Isds', 'Cocos Isds', 'Congo', 'Cook Isds', 'Cuba', 'CuraÃ§ao', 'Czechia']


In [24]:
# Manual mapping for the most important countries
name_mapping = {
    "Bolivia (Plurinational State of)": "Bolivia",
    "Bosnia Herzegovina": "Bosnia and Herzegovina",
    "Br. Virgin Isds": "British Virgin Islands",
    "Central African Rep.": "Central African Republic",
    "China, Hong Kong SAR": "Hong Kong SAR, China",
    "China, Macao SAR": "Macao SAR, China",
    "Congo": "Congo, Rep.",
    "Czechia": "Czech Republic",
    "CuraÃ§ao": "Curacao",
    "Dem. Rep. of the Congo": "Congo, Dem. Rep.",
    "Iran (Islamic Republic of)": "Iran",
    "Korea, Republic of": "Korea, Rep.",
    "Lao People's Dem. Rep.": "Lao PDR",
    "Moldova, Republic of": "Moldova",
    "Russia": "Russian Federation",
    "Syria": "Syrian Arab Republic",
    "Tanzania": "Tanzania, United Rep.",
    "Türkiye": "Turkey",
    "United Kingdom": "United Kingdom",
    "United States of America": "United States",
    "Venezuela": "Venezuela, RB",
    "Viet Nam": "Vietnam",
}

baci_reshaped["country_mapped"] = baci_reshaped["country"].replace(name_mapping)

baci_with_inflation2 = baci_reshaped.merge(
    inflation_recent[["country", "year", "Headline CPI"]],
    left_on=["country_mapped", "year"],
    right_on=["country", "year"],
    how="left"
).drop(columns=["country_y", "country_mapped"]).rename(columns={"country_x": "country"})

print(baci_with_inflation2["Headline CPI"].isna().sum(), "missing CPI values")

6301 missing CPI values


In [25]:
missing2 = baci_with_inflation2[baci_with_inflation2["Headline CPI"].isna()]["country"].unique()
print(sorted(missing2)[:30])

['American Samoa', 'Andorra', 'Anguilla', 'Bahamas', 'Bermuda', 'Bonaire', 'Br. Virgin Isds', 'Cayman Isds', 'Christmas Isds', 'Cocos Isds', 'Cook Isds', 'Cuba', 'CuraÃ§ao', "CÃ´te d'Ivoire", "Dem. People's Rep. of Korea", 'Dominican Rep.', 'Egypt', 'Eritrea', 'Eswatini', 'FS Micronesia', 'Falkland Isds (Malvinas)', 'Fr. South Antarctic Terr.', 'French Polynesia', 'Gambia', 'Gibraltar', 'Greenland', 'Guam', 'Iran', 'Kyrgyzstan', 'Marshall Isds']


In [26]:
name_mapping2 = {
    "Bolivia (Plurinational State of)": "Bolivia",
    "Bosnia Herzegovina": "Bosnia and Herzegovina",
    "Br. Virgin Isds": "British Virgin Islands",
    "Central African Rep.": "Central African Republic",
    "China, Hong Kong SAR": "Hong Kong SAR, China",
    "China, Macao SAR": "Macao SAR, China",
    "Congo": "Congo, Rep.",
    "Czechia": "Czech Republic",
    "CuraÃ§ao": "Curacao",
    "CÃ´te d'Ivoire": "Cote d'Ivoire",
    "Dem. Rep. of the Congo": "Congo, Dem. Rep.",
    "Dem. People's Rep. of Korea": "Korea, Dem. Rep.",
    "Dominican Rep.": "Dominican Republic",
    "Egypt": "Egypt, Arab Rep.",
    "Eswatini": "Eswatini",
    "FS Micronesia": "Micronesia, Fed. Sts.",
    "Gambia": "Gambia, The",
    "Iran": "Iran, Islamic Rep.",
    "Iran (Islamic Republic of)": "Iran, Islamic Rep.",
    "Korea, Republic of": "Korea, Rep.",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Lao People's Dem. Rep.": "Lao PDR",
    "Moldova, Republic of": "Moldova",
    "Russia": "Russian Federation",
    "Syria": "Syrian Arab Republic",
    "Tanzania": "Tanzania",
    "Türkiye": "Turkey",
    "United States of America": "United States",
    "Venezuela": "Venezuela, RB",
    "Viet Nam": "Vietnam",
    "Yemen": "Yemen, Rep.",
    "Bahamas": "Bahamas, The",
    "Kyrgyz Rep.": "Kyrgyz Republic",
}

baci_reshaped["country_mapped"] = baci_reshaped["country"].replace(name_mapping2)

baci_with_inflation3 = baci_reshaped.merge(
    inflation_recent[["country", "year", "Headline CPI"]],
    left_on=["country_mapped", "year"],
    right_on=["country", "year"],
    how="left"
).drop(columns=["country_y", "country_mapped"]).rename(columns={"country_x": "country"})

print(baci_with_inflation3["Headline CPI"].isna().sum(), "missing CPI values")

5769 missing CPI values


In [27]:
baci_with_inflation3.to_csv("../data/processed/baci_gold_with_inflation.csv", index=False)
print("saved!")
print(f"Total rows: {len(baci_with_inflation3)}")
print(f"Missing CPI: {baci_with_inflation3['Headline CPI'].isna().sum()} ({baci_with_inflation3['Headline CPI'].isna().mean()*100:.1f}%)")

saved!
Total rows: 50331
Missing CPI: 5769 (11.5%)
